<a href="https://colab.research.google.com/github/Lava0506/Multibot_Adventure_Game/blob/main/Module_2_Multibot_Playground.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 2: Multibot Chat Playground


In [1]:

import os, time, json, getpass
import requests
import pandas as pd
from IPython.display import display, Markdown

In [2]:

try:
    from google.colab import userdata
    API_KEY = userdata.get("OPENROUTER_API_KEY")
except Exception:
    API_KEY = None

if not API_KEY:
    API_KEY = getpass.getpass("Enter your OpenRouter API key: ")

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

In [3]:

MODELS = {
    "gpt":    "openai/gpt-4o-mini",
    "claude": "anthropic/claude-3.5-haiku",
    "gemini": "google/gemini-2.0-flash-001",
}

In [4]:
def call_model(model_key, messages, temperature=0.7, max_tokens=700, retries=2):
    """
    Send a chat request to OpenRouter and return a dict with the reply,
    latency, token usage, and any error.

    model_key   : nickname from MODELS
    messages    : list of {"role": ..., "content": ...} dicts (OpenAI chat format)
    temperature : randomness (0 = deterministic, higher = more creative)
    max_tokens  : cap on reply length (also caps cost)
    """
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODELS[model_key],
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    for attempt in range(retries + 1):
        start = time.time()
        try:
            r = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=60)
            latency = round(time.time() - start, 2)
            if r.status_code == 200:
                data = r.json()
                usage = data.get("usage", {})
                return {
                    "model": model_key,
                    "text": data["choices"][0]["message"]["content"],
                    "latency_s": latency,
                    "prompt_tokens": usage.get("prompt_tokens"),
                    "completion_tokens": usage.get("completion_tokens"),
                    "error": None,
                }
            err = f"HTTP {r.status_code}: {r.text[:200]}"
        except Exception as e:
            latency = round(time.time() - start, 2)
            err = str(e)
        time.sleep(1.5 * (attempt + 1))

    return {"model": model_key, "text": "", "latency_s": latency,
            "prompt_tokens": None, "completion_tokens": None, "error": err}

In [5]:

print(call_model("gpt", [{"role": "user", "content": "Say hello in five words."}]))

{'model': 'gpt', 'text': 'Hello! How are you today?', 'latency_s': 0.72, 'prompt_tokens': 13, 'completion_tokens': 7, 'error': None}


## The three tasks
Same prompt goes to every model so the model is the only variable.

In [6]:
SYSTEM = "You are the narrator of a text-based adventure game."

TASKS = {
    "creative": (
        "Write the opening scene (about 150 words) of a text adventure set in an abandoned "
        "lighthouse during a storm. Use second person, build tension, and end with exactly "
        "three numbered choices for the player."
    ),
    "analytical": (
        "In a text adventure, the player has a rope (10 m), a lantern (burns 30 minutes), and a key. "
        "A cellar is 12 m deep and holds a chest that needs the key. Climbing down or up takes 1 minute "
        "per 3 m of rope used, and the player can drop the lantern down first. "
        "Is the rope long enough to reach the chest? Reason step by step, state any assumptions, "
        "and give a final verdict in one sentence."
    ),
    "instruction": (
        "Return ONLY valid JSON, with no markdown fences and no commentary, describing the game state "
        "with exactly these keys: \"player_name\" (string), \"health\" (integer 0-100), "
        "\"inventory\" (array of exactly 3 strings), \"location\" (string), "
        "\"quest_complete\" (boolean). Use a lighthouse adventure theme."
    ),
}

def build_messages(prompt):
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}]

In [7]:

results = []
for task_name, prompt in TASKS.items():
    for model_key in MODELS:
        res = call_model(model_key, build_messages(prompt))
        res["task"] = task_name
        results.append(res)

df = pd.DataFrame(results)[["task", "model", "latency_s", "prompt_tokens", "completion_tokens", "error", "text"]]
display(df.drop(columns=["text"]))

,task,model,latency_s,prompt_tokens,completion_tokens,error
0,creative,gpt,5.29,63.0,221.0,None
1,creative,claude,0.07,NaN,NaN,"HTTP 404: {""error"":{""message"":""anthropic/claud..."
2,creative,gemini,0.09,NaN,NaN,"HTTP 404: {""error"":{""message"":""google/gemini-2..."
3,analytical,gpt,4.27,120.0,213.0,None
4,analytical,claude,0.08,NaN,NaN,"HTTP 404: {""error"":{""message"":""anthropic/claud..."
5,analytical,gemini,0.09,NaN,NaN,"HTTP 404: {""error"":{""message"":""google/gemini-2..."
6,instruction,gpt,1.35,91.0,50.0,None
7,instruction,claude,0.09,NaN,NaN,"HTTP 404: {""error"":{""message"":""anthropic/claud..."
8,instruction,gemini,0.08,NaN,NaN,"HTTP 404: {""error"":{""message"":""google/gemini-2..."


In [8]:

for task_name in TASKS:
    display(Markdown(f"## Task: {task_name}"))
    for _, row in df[df.task == task_name].iterrows():
        display(Markdown(f"**{row.model}** ({row.latency_s}s, {row.completion_tokens} tokens)"))
        print(row.text)
        print("-" * 70)

## Task: creative

**gpt** (5.29s, 221.0 tokens)

The wind howls like a restless spirit as you stand before the crumbling lighthouse, its once-proud tower silhouetted against the tumultuous sky. Rain lashes down, soaking you to the bone, while distant thunder rumbles like a warning. The air is thick with the scent of salt and decay, mingling with the metallic tang of the coming storm. 

You push open the rusted door, which creaks ominously on its hinges, revealing a shadowy interior cluttered with remnants of a long-abandoned life. The faint flicker of lightning illuminates the crumbling staircase that spirals upward, disappearing into darkness. A chill creeps down your spine, and you can almost hear the whispers of those who once called this place home.

Suddenly, a loud clap of thunder shakes the ground, and the door slams shut behind you.

What will you do?

1. Climb the staircase to investigate the upper levels.  
2. Search the ground floor for clues about the lighthouse's past.  
3. Try to force the door open and escape back into

**claude** (0.07s, nan tokens)


----------------------------------------------------------------------


**gemini** (0.09s, nan tokens)

... (11 outputs left out)

**claude** (0.09s, nan tokens)


----------------------------------------------------------------------


**gemini** (0.08s, nan tokens)


----------------------------------------------------------------------


## Format check for the instruction-following task
Programmatic check: does each model return parseable JSON with exactly the requested keys and types?

In [9]:
EXPECTED = {"player_name": str, "health": int, "inventory": list, "location": str, "quest_complete": bool}

def check_json(text):
    """Return (passed, reason) for the strict JSON task."""
    try:
        obj = json.loads(text.strip())
    except Exception as e:
        return False, f"not valid JSON ({type(e).__name__})"
    if set(obj) != set(EXPECTED):
        return False, f"keys differ: {sorted(obj)}"
    for k, t in EXPECTED.items():
        if not isinstance(obj[k], t) or (t is int and isinstance(obj[k], bool)):
            return False, f"wrong type for {k}"
    if not 0 <= obj["health"] <= 100:
        return False, "health out of range"
    if len(obj["inventory"]) != 3 or not all(isinstance(i, str) for i in obj["inventory"]):
        return False, "inventory must be 3 strings"
    return True, "ok"

for _, row in df[df.task == "instruction"].iterrows():
    print(row.model, "->", check_json(row.text))

gpt -> (True, 'ok')
claude -> (False, 'not valid JSON (JSONDecodeError)')
gemini -> (False, 'not valid JSON (JSONDecodeError)')


## Consistency check
Run the same prompt several times per model to see how stable the output is (here: JSON pass rate).

In [10]:
N = 3
rows = []
for model_key in MODELS:
    passes = 0
    lats = []
    for _ in range(N):
        r = call_model(model_key, build_messages(TASKS["instruction"]), temperature=0.7)
        ok, _ = check_json(r["text"]) if not r["error"] else (False, r["error"])
        passes += ok
        lats.append(r["latency_s"])
    rows.append({"model": model_key, "json_pass": f"{passes}/{N}", "avg_latency_s": round(sum(lats)/N, 2)})
display(pd.DataFrame(rows))

,model,json_pass,avg_latency_s
0,gpt,3/3,1.36
1,claude,0/3,0.07
2,gemini,0/3,0.08


## Interactive playground
Play the adventure with any model and switch models mid-game. Commands: `/model <name>`, `/models`, `/reset`, `/quit`.

In [11]:
def playground():
    current = list(MODELS)[0]
    history = [{"role": "system", "content":
        "You are the narrator of a text adventure in an abandoned lighthouse. "
        "Describe scenes in under 80 words, then offer 3 numbered choices."}]
    print(f"Playground started with model: {current}. Type /models to list, /quit to exit.")
    while True:
        user = input("You> ").strip()
        if user == "/quit":
            break
        if user == "/models":
            print(", ".join(MODELS)); continue
        if user == "/reset":
            history = history[:1]; print("Game reset."); continue
        if user.startswith("/model "):
            name = user.split(maxsplit=1)[1]
            if name in MODELS:
                current = name; print(f"Switched to {current}")
            else:
                print("Unknown model.")
            continue
        history.append({"role": "user", "content": user or "Begin the adventure."})
        res = call_model(current, history)
        if res["error"]:
            print("Error:", res["error"]); history.pop(); continue
        history.append({"role": "assistant", "content": res["text"]})
        print(f"[{current} | {res['latency_s']}s]\n{res['text']}\n")

playground()

## Simple task router
A rule-based router that picks a model per task type. Replace the mapping with whatever your own results show.

In [12]:

ROUTES = {
    "creative":    "claude",
    "analytical":  "gpt",
    "instruction": "gemini",
}
DEFAULT_ROUTE = "gpt"

def classify(prompt):
    """Very rough keyword classifier. A better version could use a cheap LLM call."""
    p = prompt.lower()
    if "json" in p or "exactly" in p or "format" in p:
        return "instruction"
    if any(w in p for w in ["story", "poem", "scene", "write"]):
        return "creative"
    if any(w in p for w in ["step by step", "calculate", "compare", "reason", "why"]):
        return "analytical"
    return None

def routed_call(prompt, fallback="gpt"):
    task = classify(prompt)
    primary = ROUTES.get(task, DEFAULT_ROUTE)
    res = call_model(primary, build_messages(prompt))
    if res["error"] and primary != fallback:
        res = call_model(fallback, build_messages(prompt))
    res["routed_as"] = task
    return res

print(routed_call("Write a short scene in a haunted library.")["model"])

gpt


# Reflection: Building a Multibot Chat Playground

For this assignment, I developed a Multibot Chat Playground using Python and OpenRouter. The main objective was to understand how different Large Language Models (LLMs) handle the same prompts and compare their performance across different tasks. I used OpenRouter as a common API platform, which allowed me to configure multiple models using a single API key instead of setting up separate API integrations for each provider.

The three models selected for the experiment were GPT-4o mini, Claude 3.5 Haiku, and Gemini 2.0 Flash. I designed three tasks to evaluate creative writing, analytical reasoning, and instruction following. Each model received the same prompt for a particular task to make the comparison fair.

The first task focused on creative writing. The models were asked to generate an opening scene for a text-based adventure game set in an abandoned lighthouse during a storm. The prompt required second-person narration, a tense atmosphere, and exactly three numbered choices at the end. GPT-4o mini successfully generated a descriptive scene with an engaging atmosphere and three choices for the player. Its response used details such as thunder, rain, a mysterious staircase, and an abandoned lighthouse to create suspense. The response took 5.29 seconds and generated 221 completion tokens. However, Claude 3.5 Haiku and Gemini 2.0 Flash returned HTTP 404 errors, so their creative writing abilities could not be evaluated in this experiment.

The second task tested analytical reasoning through a puzzle involving a 10-metre rope, a 12-metre-deep cellar, a lantern, and a locked chest. GPT-4o mini approached the problem step by step, identified the difference between the rope length and cellar depth, and correctly concluded that the rope was insufficient to reach the bottom. It also explained that the lantern did not affect the rope-length requirement. The response took 4.27 seconds and generated 213 completion tokens. This demonstrated the model's ability to break a problem into smaller parts and provide a logical explanation. The other two models again returned HTTP 404 errors, preventing a direct comparison.

The third task evaluated instruction following. The models were asked to return only valid JSON containing five specific keys: player_name, health, inventory, location, and quest_complete. Each field also had a required data type, and additional conditions were specified for the health value and inventory list. GPT-4o mini successfully returned valid JSON with all the required keys and correct value types. Its response passed the Python validation function and took 1.35 seconds, generating 50 completion tokens. In comparison, Claude and Gemini failed the JSON validation because their responses contained API error messages instead of valid JSON.

To make the evaluation more reliable, I also implemented a consistency test. The same instruction-following prompt was sent three times to each model. GPT-4o mini achieved a 3/3 JSON pass rate, with an average response time of 1.36 seconds. Claude and Gemini both achieved 0/3 because their API requests failed. Their recorded response times were 0.07 and 0.08 seconds respectively, but these values represent failed requests rather than actual generation speeds. Therefore, they cannot be considered evidence that these models are faster.

The playground was built using Python functions for API communication, message construction, model selection, and response processing. I used OpenRouter's chat completions endpoint, which accepts the selected model identifier, a messages list, temperature, and maximum token limit. The temperature was set to 0.7 to allow some variation in creative responses, while the maximum token limit was set to control response length. The notebook also records latency, prompt tokens, completion tokens, and errors. One issue encountered during testing was the HTTP 404 error for Claude and Gemini. This indicates that the requested model endpoints were not successfully available through the configured requests. Possible causes include incorrect model identifiers or model availability issues, which would need further investigation.

Another important feature of the notebook is the task-based model router. It uses keywords to identify whether a prompt is creative, analytical, or instruction-based and selects a model accordingly. This approach can help applications choose a suitable model automatically instead of relying on one model for every task. In a fully working version, the routing configuration could be updated using actual comparison results. It could also include fallback models, API error handling, and output validation to make the system more reliable.

Overall, this assignment helped me understand both the advantages and challenges of using multiple LLM providers in one application. GPT-4o mini successfully completed all three tasks and demonstrated reliable JSON formatting in repeated tests. However, because Claude and Gemini encountered API errors, I could not make a complete performance comparison or identify a definite winner across all three providers. This was an important learning experience because it showed that successful API integration is just as important as model capability. A practical multibot application should consider response quality, accuracy, formatting, latency, token usage, API availability, and reliability before selecting a model. In the future, I would improve this playground by verifying model availability, adding automatic fallback mechanisms, and repeating the experiments once all providers are working. The biggest takeaway is that different models can be useful for different tasks, but their selection should be based on actual testing and reliable results rather than assumptions.
